# Surface-Based Source Spaces

The Source Space is a distributed dipole model. Brain sources are approximated by dipoles and placed uniformly along the brain surface (more specifically, the white surface — the boundary between gray and white matter). The number of dipoles is adjustable based on the `spacing`. We then eliminate any dipole that is not located within the gray matter, as only the synaptic currents occurring in the gray matter are believed to contribute to EEG.

# Batch Wise Run

Run the code for all subjects of a given dataset.

## 📦 Imports

In [1]:
import mindmap as mm 
from pathlib import Path
import mne

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
- FREESURFER reconstruction folder.
    - white surface
- `ctx`: Cortical Mask. 
- `cras`: Translation to switch from FreeSurfer space to Subject space.

📤 **<font color=pink>Outputs</font>**
- `sources`: MNE object containing dipole locations and dipole distances to each other.

🛠️ **<font color=violet>Configuration</font>**:
- `fconfig`: Configuration file.

🛠️ **<font color=violet>Source Space</font>**:
- `spacing`: spacing between sources

🛠️ **<font color=violet>Other options</font>**:
- `n_jobs`: number of parrallel processes during source space building
- `overwrite`: If True overwrite old files. If False, skip subjects already having files.

In [11]:
########################## parameters #############################
# Configuration
fconfig = "../configurations/spike.yml"

# Source Space
spacing = 7

# Other options
overwrite = True
n_jobs    = 10
###################################################################

# config
config   = mm.io.read_config(fconfig)
mm_dir   = Path(config["MM_DIR"])
subjects = config["SUBS"]

# files & folders
fsrc   = lambda sub: mm_dir/sub/"src"/f"{sub}_src.fif"
fs_dir = Path(config["FS_DIR"])
mm_dir = Path(config["MM_DIR"])

## ♾️ Loop

In [18]:
for sub in subjects:
    
    # check file existence
    fname = fsrc(sub)
    if fname.exists() and not overwrite:
        print(f"\033[93mSurface-based source space already exists for subject {sub}\033[0m")
        continue 
    elif not fname.parent.exists():
        fname.parent.mkdir(parents=True)

    try: 
        # data
        data = mm.io.read_dataset(sub, "ctx", "cras")

        # create source space
        sources = mne.setup_source_space(
            subject=sub,
            spacing=spacing,
            surface="white",
            add_dist=True,      # geodesic distances between sources
            subjects_dir=fs_dir,
            n_jobs=n_jobs,
            verbose="warning",
        )

        # cortical constraint 
        sources = mm.utils.mask_sources(sources, data["ctx"], data["cras"])

    except Exception as e:
        print(f"\033[91mFailed to generate source space for subject {sub} due to: {e}\033[0m")
        continue

    # save
    sources.save(fname, overwrite=True, verbose="warning")
    print(f"\033[92mSuccessfully saved surface-based source space for subject {sub}\033[0m")

DATA SUMMARY
Dataset: spike
Subject: sub-pat01
Retrieved objects:
- ctx
- cras
Missing objects:
Successfully saved surface-based source space for subject sub-pat01
DATA SUMMARY
Dataset: spike
Subject: sub-pat02
Retrieved objects:
- ctx
- cras
Missing objects:
Successfully saved surface-based source space for subject sub-pat02
DATA SUMMARY
Dataset: spike
Subject: sub-pat03
Retrieved objects:
- ctx
- cras
Missing objects:
Successfully saved surface-based source space for subject sub-pat03
DATA SUMMARY
Dataset: spike
Subject: sub-pat04
Retrieved objects:
- ctx
- cras
Missing objects:
Successfully saved surface-based source space for subject sub-pat04
DATA SUMMARY
Dataset: spike
Subject: sub-pat05
Retrieved objects:
- ctx
- cras
Missing objects:
Successfully saved surface-based source space for subject sub-pat05
DATA SUMMARY
Dataset: spike
Subject: sub-pat06
Retrieved objects:
- ctx
- cras
Missing objects:
Successfully saved surface-based source space for subject sub-pat06
DATA SUMMARY
Dat

# Step Wise Run

Run the code step by step for a single subject.

## 📦 Imports

In [6]:
from pathlib import Path
import mindmap as mm
import mne

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
- FREESURFER reconstruction folder.
    - white surface
- `ctx`: Cortical Mask. 
- `cras`: Translation to switch from FreeSurfer space to Subject space.

🛠️ **<font color=violet>Configuration</font>**:
- `fconfig`: Configuration file.
- `sub`: Subject id (position in list SUBS)

In [7]:
############# Parameters ###############
fconfig = "../configurations/simf.yml"
sub     = 0
########################################

config = mm.io.read_config("../configurations/simf.yml")
sub    = config["SUBS"][sub]

# directories
fs_dir = Path(config["FS_DIR"])
mm_dir = Path(config["MM_DIR"])

# data
data = mm.io.read_dataset(sub, "ctx", "cras")

DATA SUMMARY
Dataset: simf
Subject: vepcon_sub-10
Retrieved objects:
- cras
- ctx
Missing objects:


## Source Space

#### ⚙️ **<font color=green>Generate Source Space</font>**

📤 **<font color=pink>Outputs</font>**
- `sources`: MNE object containing dipole locations and dipole distances to each other.

🛠️ **<font color=violet>Source space propeties</font>**:
- `spacing`: Spacing in [mm] between dipoles.

🛠️ **<font color=violet>Other options</font>**:
- `n_jobs`: Number of parrallel jobs

In [8]:
########## Parameters ###########
# source space 
spacing = 7

# other options
n_jobs = 10
#################################


# source space
sources = mne.setup_source_space(
    subject=sub,
    spacing=spacing,
    surface="white",
    add_dist=True,      # geodesic distances between sources
    subjects_dir=fs_dir,
    n_jobs=n_jobs,
    verbose="warning",
)
    
# gray matter constraint 
sources = mm.utils.mask_sources(sources, data["ctx"], data["cras"])

print("\033[92mSUMMARY\033[0m")
for src in sources:
    print(f"Source space id: {src["id"]}")
    print(f"Type: {src["type"]}")
    print(f"Number of sources: {src["nuse"]}\n")

SUMMARY
Source space id: 101 (FIFFV_MNE_SURF_LEFT_HEMI)
Type: surf
Number of sources: 2529

Source space id: 102 (FIFFV_MNE_SURF_RIGHT_HEMI)
Type: surf
Number of sources: 2432



🌟 **Visualize source space**

In [9]:
mm.viz.plot_sources(sources, mask=data["ctx"], cras=data["cras"], mask_opacity=0.2).show()

Widget(value='<iframe src="http://localhost:34817/index.html?ui=P_0x7036c224ac30_0&reconnect=auto" class="pyvi…

#### 💾 Save Source Space

🛠️ **<font color=violet>Options</font>**:
- `overwrite`: if True, overwrites previously saved data.

In [10]:
######### Parameters ############
overwrite = True 
#################################

fsrc   = mm_dir/sub/"src"/f"{sub}_src.fif"

if not fsrc.parent.exists():
    fsrc.parent.mkdir(parents=True)

sources.save(fsrc, overwrite=overwrite, verbose="warning")
print(f"\033[92mSuccessfully saved surface-based source space for subject {sub}\033[0m")

Successfully saved surface-based source space for subject vepcon_sub-10
